In [2]:

# ============================================================
# CELL 1: SETUP AND API CONFIGURATION
# ============================================================

!pip install -q langchain langchain-groq gspread

import os
import getpass
import pandas as pd

from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.tools import tool

from pydantic import BaseModel, Field
from typing import Literal, Optional

from google.colab import auth
import gspread
from google.auth import default

# Enter Groq API key at runtime
os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

# Initialize LLM
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_retries=3
)

print("Groq LLM initialized successfully!")


# ============================================================
# CELL 2: GOOGLE SHEETS AUTHENTICATION
# ============================================================

auth.authenticate_user()

creds, _ = default()
gc = gspread.authorize(creds)

try:
    sh = gc.open("Spice Hub Orders")
    ws = sh.sheet1
    print("Existing Google Sheet opened.")

except gspread.exceptions.SpreadsheetNotFound:
    sh = gc.create("Spice Hub Orders")
    ws = sh.sheet1
    ws.append_row([
        "customer_name",
        "item",
        "quantity",
        "order_type",
        "total"
    ])

if not ws.row_values(1):
    ws.append_row([
        "customer_name",
        "item",
        "quantity",
        "order_type",
        "total"
    ])

print("Google Sheet ready:", sh.url)


# ============================================================
# CELL 3: RESTAURANT DATA
# ============================================================

MENU = {
    "chicken biryani": 220,
    "mutton biryani": 300,
    "veg biryani": 160,
    "paneer biryani": 190,
    "veg burger": 100,
    "chicken burger": 150,
    "margherita pizza": 200,
    "farmhouse pizza": 280,
    "veg sandwich": 80,
    "chicken sandwich": 120
}

AVAILABILITY = {
    "chicken biryani": True,
    "mutton biryani": True,
    "veg biryani": True,
    "paneer biryani": True,
    "veg burger": True,
    "chicken burger": True,
    "margherita pizza": True,
    "farmhouse pizza": False,
    "veg sandwich": True,
    "chicken sandwich": True
}

HOURS = {
    "monday": "10 AM to 10 PM",
    "tuesday": "10 AM to 10 PM",
    "wednesday": "10 AM to 10 PM",
    "thursday": "10 AM to 10 PM",
    "friday": "10 AM to 10 PM",
    "saturday": "10 AM to 11 PM",
    "sunday": "10 AM to 11 PM"
}


# ============================================================
# CELL 4: PYDANTIC ORDER MODEL
# ============================================================

class Order(BaseModel):
    """A validated food order from a restaurant customer."""

    customer_name: str = Field(
        description="Customer's name"
    )

    item: Literal[
        "chicken biryani",
        "mutton biryani",
        "veg biryani",
        "paneer biryani",
        "veg burger",
        "chicken burger",
        "margherita pizza",
        "farmhouse pizza",
        "veg sandwich",
        "chicken sandwich"
    ]

    quantity: int = Field(
        description="Number of items ordered",
        ge=1
    )

    order_type: Literal[
        "dine-in",
        "takeaway"
    ]


# Flexible model for extracting incomplete messages
class OrderExtraction(BaseModel):
    """Extract available order details from customer messages."""

    customer_name: Optional[str] = None

    item: Optional[Literal[
        "chicken biryani",
        "mutton biryani",
        "veg biryani",
        "paneer biryani",
        "veg burger",
        "chicken burger",
        "margherita pizza",
        "farmhouse pizza",
        "veg sandwich",
        "chicken sandwich"
    ]] = None

    quantity: Optional[int] = Field(
        default=None,
        ge=1
    )

    order_type: Optional[
        Literal["dine-in", "takeaway"]
    ] = None


# ============================================================
# CELL 5: THREE RESTAURANT TOOLS
# ============================================================

# TOOL 1: GET PRICE
@tool
def get_price(item: str, qty: int = 1) -> str:
    """Get the total price in rupees for a food item.

    Args:
        item: Name of the food item.
        qty: Number of items required.
    """

    item = item.lower().strip()

    price = MENU.get(item)

    if price is None:
        return (
            f"{item} is not on our menu. "
            f"Available items: {', '.join(MENU)}"
        )

    if qty < 1:
        return "Quantity must be at least 1."

    return f"{qty} x {item} = Rs {price * qty}"


# TOOL 2: CHECK AVAILABILITY
@tool
def check_availability(item: str) -> str:
    """Check whether a food item is currently available.

    Args:
        item: Name of the food item.
    """

    item = item.lower().strip()

    if item not in AVAILABILITY:
        return f"{item} is not on our menu."

    if AVAILABILITY[item]:
        return f"Yes, {item} is currently available."

    return f"Sorry, {item} is currently unavailable."


# TOOL 3: LOG ORDER
@tool
def log_order(
    customer_name: str,
    item: str,
    quantity: int,
    order_type: str
) -> str:
    """Save a confirmed and validated customer order to Google Sheets.

    Args:
        customer_name: Customer's name.
        item: Selected food item.
        quantity: Number of items.
        order_type: Dine-in or takeaway.
    """

    try:
        order = Order(
            customer_name=customer_name,
            item=item.lower().strip(),
            quantity=quantity,
            order_type=order_type.lower().strip()
        )

    except Exception as e:
        return f"Could not save order: {e}"

    if not AVAILABILITY.get(order.item, False):
        return f"{order.item} is unavailable. Order not saved."

    total = MENU[order.item] * order.quantity

    ws.append_row([
        order.customer_name,
        order.item,
        order.quantity,
        order.order_type,
        total
    ])

    return (
        f"Order saved successfully! "
        f"{order.quantity} x {order.item} for "
        f"{order.customer_name}. "
        f"Order type: {order.order_type}. "
        f"Total: Rs {total}."
    )


# ============================================================
# CELL 6: STRUCTURED OUTPUT TESTING
# ============================================================

structured_llm = llm.with_structured_output(OrderExtraction)

structured_test_messages = [
    "I'm Ravi, give me 2 chicken biryanis for takeaway.",
    "My name is Sita. I want one veg burger for dine-in.",
    "This is Kumar, I need 3 margherita pizzas.",
    "I want 0 chicken biryanis. My name is Rahul.",
    "Give me two sandwiches."
]

print("\nSTRUCTURED OUTPUT TESTING")

for i, message in enumerate(structured_test_messages, 1):

    print("\nTEST", i)
    print("Customer:", message)

    try:
        result = structured_llm.invoke(message)
        print("Extracted Data:", result.model_dump())

    except Exception as e:
        print("Validation/Extraction Error:", e)


# ============================================================
# CELL 7: CREATE THE RESTAURANT AI AGENT
# ============================================================

SYSTEM = """
ROLE: You are the friendly food ordering assistant for Spice Hub Restaurant.

FACTS: We serve biryani, burgers, pizzas and sandwiches.
       Customers can choose dine-in or takeaway.

RULES:
1. Always use get_price for prices.
2. Always use check_availability before accepting an order.
3. Never guess prices or availability.
4. Ask for missing customer name, item, quantity or order type.
5. Do not accept items outside the menu.
6. Never accept quantity less than 1.
7. Use log_order only after all details are collected
   and the customer explicitly confirms the order.
8. Never save incomplete or invalid orders.
9. After saving, read the order back with its total.

FORMAT:
Under 60 words, friendly, English or Telugu matching the customer.
"""

agent = create_agent(
    model=llm,
    tools=[
        get_price,
        check_availability,
        log_order
    ],
    system_prompt=SYSTEM
)

history = []

def chat(text: str) -> str:
    global history

    try:
        result = agent.invoke({
            "messages": history + [
                {"role": "user", "content": text}
            ]
        })

        history = result["messages"]

        return history[-1].content

    except Exception as e:
        print("Agent error:", e)
        return "Sorry, I'm a little busy. Please try again."


print("Restaurant AI Agent initialized successfully!")


# ============================================================
# CELL 8: AUTOMATED TESTING USING IMPORTED TESTS.CSV
# ============================================================

from google.colab import files

print("Upload your tests.csv file")
uploaded = files.upload()

df = pd.read_csv("tests.csv")

results = []

print("\n" + "=" * 70)
print("SPICE HUB RESTAURANT - AUTOMATED CHATBOT TESTING")
print("=" * 70)

for _, row in df.iterrows():

    # Reset history for independent tests
    history = []

    print("\n" + "-" * 70)
    print("TEST ID:", row["Test ID"])
    print("CUSTOMER:", row["Customer Message"])
    print("EXPECTED:", row["Expected Behaviour"])

    response = chat(row["Customer Message"])

    print("CHATBOT:", response)

    results.append({
        "Test ID": row["Test ID"],
        "Customer Message": row["Customer Message"],
        "Expected Behaviour": row["Expected Behaviour"],
        "Actual Response": response
    })

results_df = pd.DataFrame(results)

results_df.to_csv("test_results.csv", index=False)

print("\nALL TEST CASES COMPLETED!")

display(results_df)


# ============================================================
# CELL 9: VIEW EXECUTION TRACE
# ============================================================

print("\nAGENT EXECUTION TRACE")

history = []

trace_result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the price of 2 chicken biryanis?"
        }
    ]
})

trace_messages = trace_result["messages"]

for m in trace_messages:
    m.pretty_print()

ai_messages = [
    m for m in trace_messages
    if getattr(m, "type", None) == "ai"
]

tool_call_count = sum(
    len(getattr(m, "tool_calls", []))
    for m in ai_messages
)

print("\nTotal AI model messages:", len(ai_messages))
print("Total tool calls:", tool_call_count)


# ============================================================
# CELL 10: TOOL RENAMING EXPERIMENT
# ============================================================

@tool
def tool1(item: str, qty: int = 1) -> str:
    """does stuff"""

    price = MENU.get(item.lower().strip())

    if price is None:
        return "Item not found."

    if qty < 1:
        return "Invalid quantity."

    return f"Total price: Rs {price * qty}"


test_agent = create_agent(
    model=llm,
    tools=[
        tool1,
        check_availability,
        log_order
    ],
    system_prompt="""
    You are a restaurant assistant.
    Use tool1 to answer price questions.
    Do not save any orders during this experiment.
    """
)

print("\nTOOL RENAMING EXPERIMENT")

experiment_result = test_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the price of 2 chicken biryanis?"
        }
    ]
})

for m in experiment_result["messages"]:
    m.pretty_print()

# Restore original agent
agent = create_agent(
    model=llm,
    tools=[
        get_price,
        check_availability,
        log_order
    ],
    system_prompt=SYSTEM
)

print("Original agent restored successfully!")


# ============================================================
# CELL 11: CREATE RESTAURANT FACTS TEXT FILE
# ============================================================

restaurant_facts = """
RESTAURANT NAME: Spice Hub Restaurant

1. Spice Hub is a family-friendly restaurant.
2. The restaurant serves vegetarian and non-vegetarian food.
3. Monday to Friday timings are 10 AM to 10 PM.
4. Saturday timings are 10 AM to 11 PM.
5. Sunday timings are 10 AM to 11 PM.
6. Chicken Biryani costs Rs 220.
7. Mutton Biryani costs Rs 300.
8. Veg Biryani costs Rs 160.
9. Paneer Biryani costs Rs 190.
10. Veg Burger costs Rs 100.
11. Chicken Burger costs Rs 150.
12. Margherita Pizza costs Rs 200.
13. Farmhouse Pizza costs Rs 280.
14. Veg Sandwich costs Rs 80.
15. Chicken Sandwich costs Rs 120.
16. Customers can choose dine-in or takeaway.
17. Home delivery is currently unavailable.
18. Customers can pay using cash or UPI.
19. Orders are subject to food availability.
20. Customers must confirm their order before it is recorded.
"""

with open("restaurant_facts.txt", "w") as f:
    f.write(restaurant_facts)

print("restaurant_facts.txt created successfully!")


# ============================================================
# CELL 12: FINAL OUTPUT
# ============================================================

print("\n" + "=" * 70)
print("SPICE HUB RESTAURANT CHATBOT COMPLETED")
print("=" * 70)

print("Google Sheet:", sh.url)
print("Restaurant facts: restaurant_facts.txt")
print("Test results: test_results.csv")

print("\nSAVED CUSTOMER ORDERS:")
display(pd.DataFrame(ws.get_all_records()))

print("\nAUTOMATED TEST RESULTS:")
display(results_df)


Enter your Groq API Key: ··········
Groq LLM initialized successfully!
Existing Google Sheet opened.
Google Sheet ready: https://docs.google.com/spreadsheets/d/1vfVYS2Bh8lnod4S6s0SKCqyKzBXp_PVRQ5l4tdrh44U

STRUCTURED OUTPUT TESTING

TEST 1
Customer: I'm Ravi, give me 2 chicken biryanis for takeaway.
Extracted Data: {'customer_name': 'Ravi', 'item': 'chicken biryani', 'quantity': 2, 'order_type': 'takeaway'}

TEST 2
Customer: My name is Sita. I want one veg burger for dine-in.
Extracted Data: {'customer_name': 'Sita', 'item': 'veg burger', 'quantity': 1, 'order_type': 'dine-in'}

TEST 3
Customer: This is Kumar, I need 3 margherita pizzas.
Extracted Data: {'customer_name': 'Kumar', 'item': 'margherita pizza', 'quantity': 3, 'order_type': None}

TEST 4
Customer: I want 0 chicken biryanis. My name is Rahul.
Validation/Extraction Error: Error code: 400 - {'error': {'message': 'Tool call validation failed: tool call validation failed: parameters for tool OrderExtraction did not match schema:

Saving tests.csv to tests.csv

SPICE HUB RESTAURANT - AUTOMATED CHATBOT TESTING

----------------------------------------------------------------------
TEST ID: 1
CUSTOMER: What is the price of 2 chicken biryanis?
EXPECTED: Display the correct price
CHATBOT: Sure! 2 chicken biryanis cost ₹440. Let me know your name, order type (dine‑in or takeaway), and if you’d like to place the order.

----------------------------------------------------------------------
TEST ID: 2
CUSTOMER: Is veg burger available?
EXPECTED: Check and display availability
CHATBOT: Yes, the veg burger is currently available! Would you like to place an order?

----------------------------------------------------------------------
TEST ID: 3
CUSTOMER: I want 2 chicken biryanis for takeaway.
EXPECTED: Ask for customer name
CHATBOT: Sure! May I have your name to place the order?

----------------------------------------------------------------------
TEST ID: 4
CUSTOMER: Do you sell donuts?
EXPECTED: Explain item is not 

,Test ID,Customer Message,Expected Behaviour,Actual Response
0,1,What is the price of 2 chicken biryanis?,Display the correct price,Sure! 2 chicken biryanis cost ₹440. Let me kno...
1,2,Is veg burger available?,Check and display availability,"Yes, the veg burger is currently available! Wo..."
2,3,I want 2 chicken biryanis for takeaway.,Ask for customer name,Sure! May I have your name to place the order?
3,4,Do you sell donuts?,Explain item is not on the menu,"I’m sorry, we don’t have donuts on our menu. W..."
4,5,My name is Rahul. I want 0 veg burgers for tak...,Reject invalid quantity,Hi Rahul! 🌟 I see you’d like veg burgers for t...
5,6,I'm Ravi. I want 2 chicken biryanis for takeaw...,Validate and save confirmed order,"Hi Ravi! We have biryani, burgers, pizzas, and..."
6,7,Is farmhouse pizza available?,Report unavailable item,"Sorry, farmhouse pizza is currently unavailabl..."
7,8,I want two sandwiches.,Ask for specific sandwich and missing details,Sure! Could you please share your name and whe...



AGENT EXECUTION TRACE
================================ Human Message =================================

What is the price of 2 chicken biryanis?
================================== Ai Message ==================================
Tool Calls:
  check_availability (fc_0b02df54-3f72-4561-ab73-551d10b2bc53)
 Call ID: fc_0b02df54-3f72-4561-ab73-551d10b2bc53
  Args:
    item: chicken biryani
================================= Tool Message =================================
Name: check_availability

Yes, chicken biryani is currently available.
================================== Ai Message ==================================
Tool Calls:
  get_price (fc_896f5b4c-12ac-4b64-b8c2-17fb39f43e66)
 Call ID: fc_896f5b4c-12ac-4b64-b8c2-17fb39f43e66
  Args:
    item: chicken biryani
    qty: 2
================================= Tool Message =================================
Name: get_price

2 x chicken biryani = Rs 440
================================== Ai Message ==================================

Sure! 2 chi

""



AUTOMATED TEST RESULTS:


,Test ID,Customer Message,Expected Behaviour,Actual Response
0,1,What is the price of 2 chicken biryanis?,Display the correct price,Sure! 2 chicken biryanis cost ₹440. Let me kno...
1,2,Is veg burger available?,Check and display availability,"Yes, the veg burger is currently available! Wo..."
2,3,I want 2 chicken biryanis for takeaway.,Ask for customer name,Sure! May I have your name to place the order?
3,4,Do you sell donuts?,Explain item is not on the menu,"I’m sorry, we don’t have donuts on our menu. W..."
4,5,My name is Rahul. I want 0 veg burgers for tak...,Reject invalid quantity,Hi Rahul! 🌟 I see you’d like veg burgers for t...
5,6,I'm Ravi. I want 2 chicken biryanis for takeaw...,Validate and save confirmed order,"Hi Ravi! We have biryani, burgers, pizzas, and..."
6,7,Is farmhouse pizza available?,Report unavailable item,"Sorry, farmhouse pizza is currently unavailabl..."
7,8,I want two sandwiches.,Ask for specific sandwich and missing details,Sure! Could you please share your name and whe...


In [7]:
# ============================================================
# DAY 5 - RAG + MEMORY FOR SPICE HUB RESTAURANT
# ============================================================

!pip install -q sentence-transformers faiss-cpu

# ============================================================
# 1. IMPORTS
# ============================================================

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent


# ============================================================
# 2. RESTAURANT KNOWLEDGE
# ============================================================

FAQS = [
    "Spice Hub is a family-friendly restaurant.",
    "We serve vegetarian and non-vegetarian food.",
    "Our Monday to Friday timings are 10 AM to 10 PM.",
    "Our Saturday timings are 10 AM to 11 PM.",
    "Our Sunday timings are 10 AM to 11 PM.",
    "We offer dine-in and takeaway services.",
    "Home delivery is currently unavailable.",
    "We accept cash and UPI payments.",
    "Chicken Biryani costs Rs 220.",
    "Mutton Biryani costs Rs 300.",
    "Veg Biryani costs Rs 160.",
    "Paneer Biryani costs Rs 190.",
    "Veg Burger costs Rs 100.",
    "Chicken Burger costs Rs 150.",
    "Margherita Pizza costs Rs 200.",
    "Farmhouse Pizza costs Rs 280.",
    "Veg Sandwich costs Rs 80.",
    "Chicken Sandwich costs Rs 120.",
    "All orders are subject to food availability.",
    "Our menu includes Chicken Biryani (Rs 220), Mutton Biryani (Rs 300), Veg Biryani (Rs 160), Paneer Biryani (Rs 190), Veg Burger (Rs 100), Chicken Burger (Rs 150), Margherita Pizza (Rs 200), Farmhouse Pizza (Rs 280), Veg Sandwich (Rs 80), and Chicken Sandwich (Rs 120).",
    "Customers must confirm their orders before they are recorded."
]


# ============================================================
# 3. CONVERT FAQS INTO VECTORS
# ============================================================

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    FAQS,
    convert_to_numpy=True,
    normalize_embeddings=True
)

embeddings = np.asarray(embeddings, dtype="float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("Restaurant knowledge stored successfully!")
print("Total records:", index.ntotal)


# ============================================================
# 4. FAQ SEARCH TOOL
# ============================================================

@tool
def search_faq(question: str) -> str:
    """Search restaurant information about timings, payment,
    delivery, services, menu prices and restaurant policies."""

    question_vector = model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    question_vector = np.asarray(
        question_vector,
        dtype="float32"
    )

    scores, indices = index.search(question_vector, 3)

    results = []

    for score, i in zip(scores[0], indices[0]):
        if i >= 0 and score >= 0.35:
            results.append(FAQS[i])

    if not results:
        return "The requested information is not available in the restaurant knowledge base."

    return "\n".join(results)


# ============================================================
# 5. TEST FAQ SEARCH
# ============================================================

print(search_faq.invoke("What are your restaurant timings?"))
print(search_faq.invoke("Do you accept UPI payments?"))
print(search_faq.invoke("Do you provide home delivery?"))


# ============================================================
# 6. CREATE AGENT WITH MEMORY
# ============================================================

memory = InMemorySaver()

SYSTEM = """
ROLE:
You are a friendly restaurant assistant for Spice Hub Restaurant.

RESPONSIBILITIES:
1. Answer restaurant questions using search_faq.
2. Use get_price to calculate item prices.
3. Use check_availability before accepting an order.
4. Collect customer name, item, quantity and order type.
5. Ask for confirmation before saving an order.
6. Use log_order only after explicit customer confirmation.
7. Remember the conversation within the same customer thread.

RULES:
- Never invent restaurant information.
- If information is not available, say:
  "I'll check with the restaurant owner."
- Do not assume an unavailable item is available.
- Do not save an order without confirmation.
- Be polite and keep answers simple.
"""

agent = create_agent(
    llm,
    tools=[
        get_price,
        check_availability,
        log_order,
        search_faq
    ],
    system_prompt=SYSTEM,
    checkpointer=memory
)

print("Restaurant agent with memory is ready!")


# ============================================================
# 7. CHAT FUNCTION WITH THREAD ID
# ============================================================

def chat(thread_id, text):

    response = agent.invoke(
        {
            "messages": [
                {"role": "user", "content": text}
            ]
        },
        config={
            "configurable": {
                "thread_id": thread_id
            }
        }
    )

    return response["messages"][-1].content


# ============================================================
# 8. TEST CONVERSATION MEMORY
# ============================================================

print("\nCUSTOMER 1")

print(chat("customer_1", "My name is Rahul."))
print(chat("customer_1", "What is my name?"))

print("\nCUSTOMER 2")

print(chat("customer_2", "My name is Priya."))
print(chat("customer_2", "What is my name?"))


# ============================================================
# 9. TEST RESTAURANT QUESTIONS
# ============================================================

test_questions = [
    "What are your opening hours?",
    "Do you provide home delivery?",
    "What payment methods do you accept?",
    "What is the price of chicken biryani?",
    "Do you have vegetarian food?",
    "Can I order takeaway?",
    "Do you sell sushi?"
]

print("\nRESTAURANT QUESTION TESTING")

for i, question in enumerate(test_questions):

    print("\nQuestion:", question)

    answer = chat(
        f"test_customer_{i}",
        question
    )

    print("Answer:", answer)


# ============================================================
# 10. INTERACTIVE CHAT
# ============================================================

print("\nSPICE HUB RESTAURANT CHATBOT")
print("Type 'exit' to stop.")

thread_id = "live_customer_1"

while True:

    user_input = input("\nYou: ").strip()

    if user_input.lower() == "exit":
        print("Chat ended.")
        break

    if not user_input:
        continue

    answer = chat(thread_id, user_input)

    print("\nRestaurant Assistant:", answer)


# ============================================================
# 11. FINAL STATUS
# ============================================================

print("\nDAY 5 COMPLETED!")
print("RAG knowledge:", len(FAQS), "records")
print("FAISS vectors:", index.ntotal)
print("Memory: InMemorySaver")
print("Restaurant tools: 4")
print("Agent: Ready")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Restaurant knowledge stored successfully!
Total records: 21
Our Saturday timings are 10 AM to 11 PM.
Our Monday to Friday timings are 10 AM to 10 PM.
Our Sunday timings are 10 AM to 11 PM.
We accept cash and UPI payments.
Home delivery is currently unavailable.
We offer dine-in and takeaway services.
Restaurant agent with memory is ready!

CUSTOMER 1
Hello Rahul! How can I help you today? Are you looking for menu information, opening hours, or would you like to place an order?
Your name is Rahul.

CUSTOMER 2
Hello Priya! 👋 How can I help you today? Are you looking to place an order, or do you have a question about our menu, timings, or something else?
Your name is Priya.

RESTAURANT QUESTION TESTING

Question: What are your opening hours?
Answer: Our opening hours are:

- **Monday to Friday:** 10 AM – 10 PM  
- **Saturday:** 10 AM – 11 PM  
- **Sunday:** 10 AM – 11 PM  

Let me know if you’d like to place an order or have any other questions!

Question: Do you provide home delivery?
An